# Regional analysis with NeuroGraphBench

Reference notebook for the **regional-analysis** capability described in Section 2.2 of the NGB manuscript. It walks through, end-to-end, how to:

1. Browse neuropils, pick a region of interest, and load its mesh.
2. List subregions (e.g., LOP layers) and the neurons / celltypes arborizing in them.
3. Retrieve a per-region synapse table and visualize where synapses live.
4. Compute region-restricted connectivity, arborization, and innervation matrices.

The running example is the **lobula plate (LOP)** — specifically how **LPLC2** dendrites distribute across the four LOP layers, the structural finding that anchors the looming-detection circuit analysis in the manuscript.


## Prerequisites

1. A NeuroArch (neo4j) instance loaded with an optic-lobe dataset that includes LOP layer subregions and LPLC2 neurons (e.g., the Hemibrain or Optic Lobe connectomes used in the manuscript).
2. The FBA server running against that neo4j instance:

   ```bash
   uvicorn fba.server.app:app --reload
   ```

3. This notebook runs in the `ngb` conda env (Python 3.10 with `fba`/`ngb` installed editable).


In [ ]:
# imports
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt

from fba.client import Client

# connect to the FBA server
fba = Client("http://localhost:8000")
fba.listNeuropils()[:10]  # sanity check


## 1. Choose a region

`Client.neuropil(name)` returns a `Region` object — a lazy handle for the named neuropil. The mesh and other relationships (neurons, celltypes, subregions, synapses) are fetched on first access and cached.


In [ ]:
lop = fba.neuropil("LOP")
print(lop)
print("mesh:", type(lop.mesh).__name__, "with", len(lop.mesh.vertices), "vertices")


### Visualize LOP

The mesh is a `navis.Volume`. We can pass it straight to plotly for a quick 3D look. (A later release will provide `ngb.canvas.Canvas` to wrap this boilerplate.)

In [ ]:
def plotMesh(mesh, opacity=0.3, color="lightblue", name=None, fig=None):
    fig = fig or go.Figure()
    fig.add_trace(go.Mesh3d(
        x=mesh.vertices[:, 0], y=mesh.vertices[:, 1], z=mesh.vertices[:, 2],
        i=mesh.faces[:, 0], j=mesh.faces[:, 1], k=mesh.faces[:, 2],
        opacity=opacity, color=color, name=name or mesh.name, showlegend=True,
    ))
    fig.update_layout(width=700, height=600, scene_aspectmode="data")
    return fig

plotMesh(lop.mesh, color="lightblue", name="LOP")


## 2. Subregions — LOP layers

LOP is anatomically subdivided into four parallel layers (LOP-1 through LOP-4) that correspond to direction-selective inputs from T4/T5 neurons. `region.subregions` returns a list of `Region` objects, one per subregion mesh.


In [ ]:
layers = lop.subregions
[layer.name for layer in layers]


In [ ]:
# all four layers overlaid
fig = go.Figure()
palette = ["#1f77b4", "#2ca02c", "#ff7f0e", "#d62728"]
for layer, color in zip(layers, palette):
    plotMesh(layer.mesh, opacity=0.4, color=color, name=layer.name, fig=fig)
fig


## 3. Who innervates LOP?

`region.neurons` and `region.celltypes` return the names of neurons / distinct celltypes arborizing in the region. Returned as plain lists of strings — for richer objects we'd use the (forthcoming) `Neuron` and `Celltype` classes.


In [ ]:
celltypes = lop.celltypes
print(len(celltypes), "celltypes arborize in LOP")
print("sample:", celltypes[:10])


In [ ]:
# filter the LOP neuron list to LPLC2 cells
all_neurons = lop.neurons
lplc2_names = [n for n in all_neurons if n.startswith("LPLC2_")]
print(len(lplc2_names), "LPLC2 neurons innervate LOP:", lplc2_names[:5], "...")


## 4. Synapses in the region

`region.synapseTable()` returns a pandas DataFrame of every synapse in LOP, with `pre_name` / `post_name` / `pre_celltype` / `post_celltype` / `neurotransmitters` and 3D coordinates. The table is cached server-side per neuropil.


In [ ]:
synapses = lop.synapseTable()
synapses.head()


In [ ]:
# keep only synapses where LPLC2 is presynaptic
lplc2_synapses = synapses[synapses["pre_celltype"] == "LPLC2"]
print(len(lplc2_synapses), "LPLC2-presynaptic synapses in LOP")
lplc2_synapses["post_celltype"].value_counts().head(10)


In [ ]:
# 3D scatter of LPLC2 presynaptic sites inside LOP
fig = plotMesh(lop.mesh, opacity=0.2, color="lightgray", name="LOP")
fig.add_trace(go.Scatter3d(
    x=lplc2_synapses["x"], y=lplc2_synapses["y"], z=lplc2_synapses["z"],
    mode="markers", marker=dict(size=2, opacity=0.6, color="red"),
    name="LPLC2 pre",
))
fig


## 5. Connectivity within the region

`region.connectivity(names, entity, layout)` aggregates the region's synapse table into either a long-format (table) or pivoted (matrix) connectivity view. Pass `entity="celltype"` to roll up over individual neurons.


In [ ]:
ct_matrix = lop.connectivity(entity="celltype", layout="matrix")
ct_matrix.shape


In [ ]:
# focus on LPLC2 outputs (LPLC2 as pre, all celltypes as post)
lplc2_outputs = ct_matrix.loc["LPLC2"].sort_values(ascending=False).head(10)
lplc2_outputs.plot.bar(figsize=(10, 4), title="LPLC2 → ? in LOP (synapse count)")
plt.ylabel("synapse count"); plt.tight_layout()


## 6. Per-layer arborization

To replicate the manuscript's finding that LPLC2 dendrites span all four LOP layers, iterate over `lop.subregions` and call `layer.arborization(names=lplc2_names, entity="neuron")` for each layer. Result: a DataFrame of synapse-counts per (LPLC2 neuron, layer).


In [ ]:
per_layer = {}
for layer in layers:
    arb = layer.arborization(names=lplc2_names, entity="neuron")
    # `inputs` aggregates pre-synaptic count (NOTE: original FBA naming — see arborization docstring).
    per_layer[layer.name] = arb.set_index("name")["inputs"]

per_layer_df = pd.DataFrame(per_layer).fillna(0).astype(int)
per_layer_df.head()


In [ ]:
# heatmap: rows = LPLC2 neurons, columns = LOP layers
plt.figure(figsize=(6, max(6, 0.18 * len(per_layer_df))))
sns.heatmap(per_layer_df, cmap="viridis", cbar_kws={"label": "presyn count"})
plt.title("LPLC2 presynaptic counts across LOP layers"); plt.tight_layout()


## 7. Innervation across layers

`Client.getInnervation(names, regions=...)` counts **neurites** (skeleton nodes) of each neuron that fall inside each region's mesh — complementary to arborization, which counts synapses.


In [ ]:
layer_meshes = {layer.name: layer.mesh for layer in layers}
inn = fba.getInnervation(names=lplc2_names, entity="neuron", regions=layer_meshes, layout="matrix")
inn.head()


In [ ]:
plt.figure(figsize=(6, max(6, 0.18 * len(inn))))
sns.heatmap(inn, cmap="magma", cbar_kws={"label": "neurite count"})
plt.title("LPLC2 neurite counts in LOP layers"); plt.tight_layout()


## Summary

In ~20 cells we have:
- enumerated a neuropil and its anatomical subdivisions (`Region.subregions`),
- listed innervating neurons / celltypes (`Region.neurons`, `Region.celltypes`),
- retrieved a per-region synapse table (`Region.synapseTable()`),
- computed region-restricted connectivity, arborization, and innervation matrices.

For the LPLC2 → LOP example, the per-layer arborization and innervation heatmaps show that LPLC2 dendrites tile all four LOP layers — the structural feature that, together with PVLP011 inhibitory normalization (analyzed in the topographic-mapping notebook) and the Giant Fiber's compartmentalized dendritic readout (skeletal analysis), underpins the two-readout looming code described in the manuscript.

**Next reference notebooks:**
- `02-skeletal-analysis.ipynb` — local geodesic neighborhoods on neuron skeletons.
- `03-topographic-mapping.ipynb` — interactive 2D projection of neuron tracts.
- `04-retinotopic-mapping.ipynb` — hexagonal columnar coordinate propagation.
